# Vendor Performance & Profitability Analysis
## Notebook 03 — Exploratory Data Analysis

This notebook performs exploratory data analysis for the
Vendor Performance & Profitability Analysis project.

### Analysis Scope

- Initial Data Quality EDA
- Vendor Profitability Analysis
- Vendor Concentration Analysis
- Sales vs Profit Contribution
- Profitability Analysis
- Operational Efficiency Analysis
- Brand-Level Analysis
- Description-Level Analysis

### Objective

To identify vendor profitability patterns, operational efficiency,
sales concentration, and product-level opportunities using
Python, Pandas, SQLAlchemy, and SQLite.

In [2]:
# 1.============================================================
# VENDOR PERFORMANCE & PROFITABILITY ANALYSIS
# NOTEBOOK 03 — EXPLORATORY DATA ANALYSIS
# ============================================================

import os
import numpy as np
import pandas as pd

from sqlalchemy import create_engine

In [3]:
# ============================================================
# 2. EDA INITIALIZATION
# ============================================================

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("EDA environment initialized.")
print(f"Python version: {pd.__version__} pandas")

EDA environment initialized.
Python version: 2.3.3 pandas


In [13]:
# ============================================================
# 3. PROJECT & DATABASE CONFIGURATION
# ============================================================

PROJECT_PATH = r"D:\Kuliah\Practice\Data Analyst Portfolio\Vendor Performance Data Analytics End-To-End Project  SQL + Python + Power BI + Reporting\Vendor_Analysis"

DB_PATH = os.path.join(
    PROJECT_PATH,
    "notebooks",
    "inventory.db"
)

print("Project path:")
print(PROJECT_PATH)

print("\nDatabase path:")
print(DB_PATH)

print("\nDatabase exists:", os.path.exists(DB_PATH))

Project path:
D:\Kuliah\Practice\Data Analyst Portfolio\Vendor Performance Data Analytics End-To-End Project  SQL + Python + Power BI + Reporting\Vendor_Analysis

Database path:
D:\Kuliah\Practice\Data Analyst Portfolio\Vendor Performance Data Analytics End-To-End Project  SQL + Python + Power BI + Reporting\Vendor_Analysis\notebooks\inventory.db

Database exists: True


In [14]:
# ============================================================
# 4. DATABASE CONNECTION
# ============================================================

engine = create_engine(f"sqlite:///{DB_PATH}")

print("SQLite database connection created successfully.")

SQLite database connection created successfully.


In [15]:
# ============================================================
# 5. DATABASE TABLE INVENTORY
# ============================================================

tables_query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables = pd.read_sql_query(tables_query, engine)

print("Available tables:")
display(tables)

Available tables:


,name
0,begin_inventory
1,end_inventory
2,purchase_prices
3,purchases
4,sales
5,vendor_invoice
6,vendor_sales_summary


In [16]:
# ============================================================
# 6. TABLE ROW COUNTS
# ============================================================

table_counts = []

for table_name in tables["name"]:
    query = f'SELECT COUNT(*) AS row_count FROM "{table_name}"'
    row_count = pd.read_sql_query(query, engine).iloc[0, 0]

    table_counts.append({
        "Table": table_name,
        "RowCount": row_count
    })

table_counts = pd.DataFrame(table_counts)

display(table_counts)

,Table,RowCount
0,begin_inventory,206529
1,end_inventory,224489
2,purchase_prices,12261
3,purchases,2372474
4,sales,12825363
5,vendor_invoice,5543
6,vendor_sales_summary,10692


In [17]:
# ============================================================
# 7. DATASET SCHEMA OVERVIEW
# ============================================================

for table_name in tables["name"]:
    print(f"\n{'=' * 70}")
    print(f"TABLE: {table_name}")
    print(f"{'=' * 70}")

    schema_query = f'PRAGMA table_info("{table_name}")'
    schema = pd.read_sql_query(schema_query, engine)

    display(
        schema[["name", "type"]]
        .rename(columns={
            "name": "Column",
            "type": "DataType"
        })
    )


TABLE: begin_inventory


,Column,DataType
0,InventoryId,TEXT
1,Store,BIGINT
2,City,TEXT
3,Brand,BIGINT
4,Description,TEXT
5,Size,TEXT
6,onHand,BIGINT
7,Price,FLOAT
8,startDate,TEXT



TABLE: end_inventory


,Column,DataType
0,InventoryId,TEXT
1,Store,BIGINT
2,City,TEXT
3,Brand,BIGINT
4,Description,TEXT
5,Size,TEXT
6,onHand,BIGINT
7,Price,FLOAT
8,endDate,TEXT



TABLE: purchase_prices


...

,Column,DataType
0,InventoryId,TEXT
1,Store,BIGINT
2,Brand,BIGINT
3,Description,TEXT
4,Size,TEXT
5,SalesQuantity,BIGINT
6,SalesDollars,FLOAT
7,SalesPrice,FLOAT
8,SalesDate,TEXT
9,Volume,FLOAT



TABLE: vendor_invoice


,Column,DataType
0,VendorNumber,BIGINT
1,VendorName,TEXT
2,InvoiceDate,TEXT
3,PONumber,BIGINT
4,PODate,TEXT
5,PayDate,TEXT
6,Quantity,BIGINT
7,Dollars,FLOAT
8,Freight,FLOAT
9,Approval,TEXT



TABLE: vendor_sales_summary


,Column,DataType
0,VendorNumber,INTEGER
1,VendorName,TEXT
2,Brand,INTEGER
3,Description,TEXT
4,PurchasePrice,REAL
5,ActualPrice,REAL
6,Volume,REAL
7,TotalPurchaseQuantity,INTEGER
8,TotalPurchaseDollars,REAL
9,TotalSalesQuantity,REAL


In [18]:
# ============================================================
# 8. MISSING-VALUE ANALYSIS
# ============================================================

missing_query = """
SELECT
    COUNT(*) AS TotalRows,
    SUM(CASE WHEN VendorNumber IS NULL THEN 1 ELSE 0 END) AS MissingVendorNumber,
    SUM(CASE WHEN VendorName IS NULL THEN 1 ELSE 0 END) AS MissingVendorName,
    SUM(CASE WHEN Brand IS NULL THEN 1 ELSE 0 END) AS MissingBrand,
    SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS MissingDescription,
    SUM(CASE WHEN PurchasePrice IS NULL THEN 1 ELSE 0 END) AS MissingPurchasePrice,
    SUM(CASE WHEN ActualPrice IS NULL THEN 1 ELSE 0 END) AS MissingActualPrice,
    SUM(CASE WHEN Volume IS NULL THEN 1 ELSE 0 END) AS MissingVolume,
    SUM(CASE WHEN TotalPurchaseQuantity IS NULL THEN 1 ELSE 0 END) AS MissingPurchaseQuantity,
    SUM(CASE WHEN TotalPurchaseDollars IS NULL THEN 1 ELSE 0 END) AS MissingPurchaseDollars,
    SUM(CASE WHEN TotalSalesQuantity IS NULL THEN 1 ELSE 0 END) AS MissingSalesQuantity,
    SUM(CASE WHEN TotalSalesDollars IS NULL THEN 1 ELSE 0 END) AS MissingSalesDollars,
    SUM(CASE WHEN TotalSalesPrice IS NULL THEN 1 ELSE 0 END) AS MissingTotalSalesPrice,
    SUM(CASE WHEN TotalExciseTax IS NULL THEN 1 ELSE 0 END) AS MissingTotalExciseTax,
    SUM(CASE WHEN FreightCost IS NULL THEN 1 ELSE 0 END) AS MissingFreightCost,
    SUM(CASE WHEN GrossProfit IS NULL THEN 1 ELSE 0 END) AS MissingGrossProfit,
    SUM(CASE WHEN StockTurnover IS NULL THEN 1 ELSE 0 END) AS MissingStockTurnover,
    SUM(CASE WHEN SalesToPurchaseRatio IS NULL THEN 1 ELSE 0 END) AS MissingSalesToPurchaseRatio,
    SUM(CASE WHEN ProfitMargin IS NULL THEN 1 ELSE 0 END) AS MissingProfitMargin
FROM vendor_sales_summary;
"""

missing_summary = pd.read_sql_query(missing_query, engine)

display(missing_summary.T.rename(columns={0: "MissingCount"}))

,MissingCount
TotalRows,10692
MissingVendorNumber,0
MissingVendorName,0
MissingBrand,0
MissingDescription,0
MissingPurchasePrice,0
MissingActualPrice,0
MissingVolume,0
MissingPurchaseQuantity,0
MissingPurchaseDollars,0


In [19]:
# ============================================================
# 9. MISSING-VALUE RELATIONSHIP
# ============================================================

missing_relationship_query = """
SELECT
    COUNT(*) AS TotalRows,
    SUM(
        CASE
            WHEN TotalSalesDollars = 0 THEN 1
            ELSE 0
        END
    ) AS ZeroSalesRows,
    SUM(
        CASE
            WHEN TotalSalesDollars = 0
                 AND ProfitMargin IS NULL
            THEN 1
            ELSE 0
        END
    ) AS ZeroSalesWithMissingMargin,
    SUM(
        CASE
            WHEN TotalSalesDollars > 0
                 AND ProfitMargin IS NULL
            THEN 1
            ELSE 0
        END
    ) AS PositiveSalesWithMissingMargin
FROM vendor_sales_summary;
"""

missing_relationship = pd.read_sql_query(
    missing_relationship_query,
    engine
)

display(missing_relationship)

,TotalRows,ZeroSalesRows,ZeroSalesWithMissingMargin,PositiveSalesWithMissingMargin
0,10692,0,0,0


In [20]:
# ============================================================
# 10. MISSING SALES VALUE VALIDATION
# ============================================================

sales_missing_query = """
SELECT
    COUNT(*) AS TotalRows,
    SUM(
        CASE
            WHEN TotalSalesDollars IS NULL THEN 1
            ELSE 0
        END
    ) AS NullSalesDollars,
    SUM(
        CASE
            WHEN TotalSalesDollars IS NULL
                 AND ProfitMargin IS NULL
            THEN 1
            ELSE 0
        END
    ) AS NullSalesWithMissingMargin,
    SUM(
        CASE
            WHEN TotalSalesDollars IS NOT NULL
                 AND ProfitMargin IS NULL
            THEN 1
            ELSE 0
        END
    ) AS NonNullSalesWithMissingMargin
FROM vendor_sales_summary;
"""

sales_missing = pd.read_sql_query(
    sales_missing_query,
    engine
)

display(sales_missing)

,TotalRows,NullSalesDollars,NullSalesWithMissingMargin,NonNullSalesWithMissingMargin
0,10692,178,178,0


In [21]:
# ============================================================
# 11. DUPLICATE ANALYSIS
# ============================================================

duplicate_query = """
SELECT
    COUNT(*) AS TotalRows,
    COUNT(DISTINCT
        VendorNumber || '|' ||
        CAST(Brand AS TEXT) || '|' ||
        Description
    ) AS UniqueVendorBrandDescription
FROM vendor_sales_summary;
"""

duplicate_summary = pd.read_sql_query(
    duplicate_query,
    engine
)

display(duplicate_summary)

,TotalRows,UniqueVendorBrandDescription
0,10692,10692


In [22]:
# ============================================================
# 12. DATE RANGE ANALYSIS
# ============================================================

date_range_queries = {
    "begin_inventory": """
        SELECT
            MIN(startDate) AS MinDate,
            MAX(startDate) AS MaxDate
        FROM begin_inventory;
    """,

    "end_inventory": """
        SELECT
            MIN(endDate) AS MinDate,
            MAX(endDate) AS MaxDate
        FROM end_inventory;
    """,

    "purchases": """
        SELECT
            MIN(PODate) AS MinDate,
            MAX(PODate) AS MaxDate
        FROM purchases;
    """,

    "sales": """
        SELECT
            MIN(SalesDate) AS MinDate,
            MAX(SalesDate) AS MaxDate
        FROM sales;
    """
}

date_ranges = []

for table_name, query in date_range_queries.items():
    result = pd.read_sql_query(query, engine)

    date_ranges.append({
        "Table": table_name,
        "MinDate": result.loc[0, "MinDate"],
        "MaxDate": result.loc[0, "MaxDate"]
    })

date_ranges = pd.DataFrame(date_ranges)

display(date_ranges)

,Table,MinDate,MaxDate
0,begin_inventory,2024-01-01,2024-01-01
1,end_inventory,2024-12-31,2024-12-31
2,purchases,2023-12-20,2024-12-23
3,sales,2024-01-01,2024-12-31


In [23]:
# ============================================================
# 13. KEY FIELD QUALITY ANALYSIS
# ============================================================

key_field_queries = {
    "purchases": {
        "VendorNumber": "VendorNumber",
        "Brand": "Brand",
        "Store": "Store",
        "InventoryId": "InventoryId"
    },
    "sales": {
        "VendorNo": "VendorNo",
        "Brand": "Brand",
        "Store": "Store",
        "InventoryId": "InventoryId"
    },
    "begin_inventory": {
        "Brand": "Brand",
        "Store": "Store",
        "InventoryId": "InventoryId"
    },
    "end_inventory": {
        "Brand": "Brand",
        "Store": "Store",
        "InventoryId": "InventoryId"
    }
}

key_quality = []

for table_name, fields in key_field_queries.items():

    for field_label, field_name in fields.items():

        query = f"""
        SELECT
            COUNT(*) AS TotalRows,
            SUM(
                CASE
                    WHEN "{field_name}" IS NULL THEN 1
                    ELSE 0
                END
            ) AS MissingValues
        FROM "{table_name}";
        """

        result = pd.read_sql_query(query, engine).iloc[0]

        key_quality.append({
            "Table": table_name,
            "Field": field_label,
            "TotalRows": int(result["TotalRows"]),
            "MissingValues": int(result["MissingValues"])
        })

key_quality = pd.DataFrame(key_quality)

display(key_quality)

,Table,Field,TotalRows,MissingValues
0,purchases,VendorNumber,2372474,0
1,purchases,Brand,2372474,0
2,purchases,Store,2372474,0
3,purchases,InventoryId,2372474,0
4,sales,VendorNo,12825363,0
5,sales,Brand,12825363,0
6,sales,Store,12825363,0
7,sales,InventoryId,12825363,0
8,begin_inventory,Brand,206529,0
9,begin_inventory,Store,206529,0


In [24]:
# ============================================================
# 14. NUMERIC DATA QUALITY
# ============================================================

numeric_fields = {
    "purchases": [
        "PurchasePrice",
        "Quantity",
        "Dollars"
    ],
    "sales": [
        "SalesQuantity",
        "SalesDollars",
        "SalesPrice",
        "Volume",
        "ExciseTax"
    ],
    "vendor_invoice": [
        "Quantity",
        "Dollars",
        "Freight"
    ]
}

numeric_quality = []

for table_name, fields in numeric_fields.items():

    for field_name in fields:

        query = f"""
        SELECT
            COUNT(*) AS TotalRows,
            SUM(CASE WHEN "{field_name}" IS NULL THEN 1 ELSE 0 END) AS NullCount,
            SUM(CASE WHEN "{field_name}" = 0 THEN 1 ELSE 0 END) AS ZeroCount,
            SUM(CASE WHEN "{field_name}" < 0 THEN 1 ELSE 0 END) AS NegativeCount
        FROM "{table_name}";
        """

        result = pd.read_sql_query(query, engine).iloc[0]

        numeric_quality.append({
            "Table": table_name,
            "Field": field_name,
            "TotalRows": int(result["TotalRows"]),
            "NullCount": int(result["NullCount"]),
            "ZeroCount": int(result["ZeroCount"]),
            "NegativeCount": int(result["NegativeCount"])
        })

numeric_quality = pd.DataFrame(numeric_quality)

display(numeric_quality)

,Table,Field,TotalRows,NullCount,ZeroCount,NegativeCount
0,purchases,PurchasePrice,2372474,0,153,0
1,purchases,Quantity,2372474,0,0,0
2,purchases,Dollars,2372474,0,153,0
3,sales,SalesQuantity,12825363,0,0,0
4,sales,SalesDollars,12825363,0,55,0
5,sales,SalesPrice,12825363,0,55,0
6,sales,Volume,12825363,0,0,0
7,sales,ExciseTax,12825363,0,0,0
8,vendor_invoice,Quantity,5543,0,0,0
9,vendor_invoice,Dollars,5543,0,0,0


In [25]:
# ============================================================
# 15. DESCRIPTIVE STATISTICS
# ============================================================

descriptive_query = """
SELECT
    COUNT(*) AS Count,
    MIN(TotalPurchaseDollars) AS MinPurchaseDollars,
    MAX(TotalPurchaseDollars) AS MaxPurchaseDollars,
    AVG(TotalPurchaseDollars) AS AvgPurchaseDollars,

    MIN(TotalSalesDollars) AS MinSalesDollars,
    MAX(TotalSalesDollars) AS MaxSalesDollars,
    AVG(TotalSalesDollars) AS AvgSalesDollars,

    MIN(GrossProfit) AS MinGrossProfit,
    MAX(GrossProfit) AS MaxGrossProfit,
    AVG(GrossProfit) AS AvgGrossProfit,

    MIN(ProfitMargin) AS MinProfitMargin,
    MAX(ProfitMargin) AS MaxProfitMargin,
    AVG(ProfitMargin) AS AvgProfitMargin,

    MIN(StockTurnover) AS MinStockTurnover,
    MAX(StockTurnover) AS MaxStockTurnover,
    AVG(StockTurnover) AS AvgStockTurnover,

    MIN(SalesToPurchaseRatio) AS MinSalesToPurchaseRatio,
    MAX(SalesToPurchaseRatio) AS MaxSalesToPurchaseRatio,
    AVG(SalesToPurchaseRatio) AS AvgSalesToPurchaseRatio

FROM vendor_sales_summary;
"""

descriptive_stats = pd.read_sql_query(
    descriptive_query,
    engine
)

display(descriptive_stats.T.rename(columns={0: "Value"}))

,Value
Count,"10,692.0000"
MinPurchaseDollars,0.7100
MaxPurchaseDollars,"3,811,251.6000"
AvgPurchaseDollars,"30,106.6934"
MinSalesDollars,1.9800
MaxSalesDollars,"5,101,919.5100"
AvgSalesDollars,"42,954.1738"
MinGrossProfit,"-52,002.7800"
MaxGrossProfit,"1,290,667.9100"
AvgGrossProfit,"12,364.6188"


In [26]:
# ============================================================
# 16. LOAD VENDOR SALES SUMMARY
# ============================================================

query = """
SELECT *
FROM vendor_sales_summary;
"""

vendor_sales_summary = pd.read_sql_query(
    query,
    engine
)

print(
    "vendor_sales_summary shape:",
    vendor_sales_summary.shape
)

display(vendor_sales_summary.head())

vendor_sales_summary shape: (10692, 18)


,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,StockTurnover,SalesToPurchaseRatio,ProfitMargin
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.2700,36.9900,"1,750.0000",145080,"3,811,251.6000","142,049.0000","5,101,919.5100","672,819.3100","260,999.2000","68,601.6800","1,290,667.9100",0.9791,1.3386,0.2530
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.1900,28.9900,"1,750.0000",164038,"3,804,041.2200","160,247.0000","4,819,073.4900","561,512.3700","294,438.6600","144,929.2400","1,015,032.2700",0.9769,1.2668,0.2106
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.2400,24.9900,"1,750.0000",187407,"3,418,303.6800","187,140.0000","4,538,120.6000","461,140.1500","343,854.0700","123,780.2200","1,119,816.9200",0.9986,1.3276,0.2468
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.1700,22.9900,"1,750.0000",201682,"3,261,197.9400","200,412.0000","4,475,972.8800","420,050.0100","368,242.8000","257,032.0700","1,214,774.9400",0.9937,1.3725,0.2714
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.8900,29.9900,"1,750.0000",138109,"3,023,206.0100","135,838.0000","4,223,107.6200","545,778.2800","249,587.8300","257,032.0700","1,199,901.6100",0.9836,1.3969,0.2841


In [28]:
# ============================================================
# 17. PREPARE VENDOR PERFORMANCE DATASET
# ============================================================

required_columns = [
    "VendorNumber",
    "VendorName",
    "TotalSalesDollars",
    "GrossProfit",
    "ProfitMargin",
    "StockTurnover",
    "SalesToPurchaseRatio"
]

vendor_performance = (
    vendor_sales_summary[
        required_columns
    ]
    .drop_duplicates()
    .copy()
)

print(
    "vendor_performance shape:",
    vendor_performance.shape
)

display(vendor_performance.head())

vendor_performance shape: (10547, 7)


,VendorNumber,VendorName,TotalSalesDollars,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio
0,1128,BROWN-FORMAN CORP,"5,101,919.5100","1,290,667.9100",0.2530,0.9791,1.3386
1,4425,MARTIGNETTI COMPANIES,"4,819,073.4900","1,015,032.2700",0.2106,0.9769,1.2668
2,17035,PERNOD RICARD USA,"4,538,120.6000","1,119,816.9200",0.2468,0.9986,1.3276
3,3960,DIAGEO NORTH AMERICA INC,"4,475,972.8800","1,214,774.9400",0.2714,0.9937,1.3725
4,3960,DIAGEO NORTH AMERICA INC,"4,223,107.6200","1,199,901.6100",0.2841,0.9836,1.3969


In [29]:
# ============================================================
# 18. VENDOR PROFITABILITY ANALYSIS
# ============================================================

columns = [
    "VendorNumber",
    "VendorName",
    "TotalSalesDollars",
    "GrossProfit",
    "ProfitMargin"
]

vendor_profitability = (
    vendor_performance[
        columns
    ]
    .sort_values(
        "TotalSalesDollars",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "vendor_profitability shape:",
    vendor_profitability.shape
)

display(vendor_profitability.head(10))

vendor_profitability shape: (10547, 5)


,VendorNumber,VendorName,TotalSalesDollars,GrossProfit,ProfitMargin
0,1128,BROWN-FORMAN CORP,"5,101,919.5100","1,290,667.9100",0.2530
1,4425,MARTIGNETTI COMPANIES,"4,819,073.4900","1,015,032.2700",0.2106
2,17035,PERNOD RICARD USA,"4,538,120.6000","1,119,816.9200",0.2468
3,3960,DIAGEO NORTH AMERICA INC,"4,475,972.8800","1,214,774.9400",0.2714
4,3960,DIAGEO NORTH AMERICA INC,"4,223,107.6200","1,199,901.6100",0.2841
5,480,BACARDI USA INC,"3,383,912.4000","917,276.4700",0.2711
6,17035,PERNOD RICARD USA,"2,773,367.7300","596,082.6500",0.2149
7,3960,DIAGEO NORTH AMERICA INC,"2,640,491.1900","736,751.6400",0.2790
8,3960,DIAGEO NORTH AMERICA INC,"2,592,041.3500","503,706.5100",0.1943
9,12546,JIM BEAM BRANDS COMPANY,"2,435,393.3900","661,966.2500",0.2718


In [30]:
# ============================================================
# 19. VENDOR CONCENTRATION ANALYSIS
# ============================================================

vendor_concentration = (
    vendor_performance
    .sort_values(
        "TotalSalesDollars",
        ascending=False
    )
    .reset_index(drop=True)
    .copy()
)

total_sales = (
    vendor_concentration[
        "TotalSalesDollars"
    ].sum()
)

total_profit = (
    vendor_concentration[
        "GrossProfit"
    ].sum()
)

vendor_concentration["SalesContribution"] = (
    vendor_concentration["TotalSalesDollars"]
    / total_sales
)

vendor_concentration["ProfitContribution"] = (
    vendor_concentration["GrossProfit"]
    / total_profit
)

vendor_concentration["CumulativeSalesContribution"] = (
    vendor_concentration["SalesContribution"]
    .cumsum()
)

vendor_concentration["CumulativeProfitContribution"] = (
    vendor_concentration["ProfitContribution"]
    .cumsum()
)

print(
    "vendor_concentration shape:",
    vendor_concentration.shape
)

display(vendor_concentration.head(10))

vendor_concentration shape: (10547, 11)


,VendorNumber,VendorName,TotalSalesDollars,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio,SalesContribution,ProfitContribution,CumulativeSalesContribution,CumulativeProfitContribution
0,1128,BROWN-FORMAN CORP,"5,101,919.5100","1,290,667.9100",0.2530,0.9791,1.3386,0.0113,0.0099,0.0113,0.0099
1,4425,MARTIGNETTI COMPANIES,"4,819,073.4900","1,015,032.2700",0.2106,0.9769,1.2668,0.0107,0.0078,0.0220,0.0177
2,17035,PERNOD RICARD USA,"4,538,120.6000","1,119,816.9200",0.2468,0.9986,1.3276,0.0100,0.0086,0.0320,0.0263
3,3960,DIAGEO NORTH AMERICA INC,"4,475,972.8800","1,214,774.9400",0.2714,0.9937,1.3725,0.0099,0.0093,0.0419,0.0357
4,3960,DIAGEO NORTH AMERICA INC,"4,223,107.6200","1,199,901.6100",0.2841,0.9836,1.3969,0.0094,0.0092,0.0513,0.0449
5,480,BACARDI USA INC,"3,383,912.4000","917,276.4700",0.2711,1.0220,1.3719,0.0075,0.0071,0.0588,0.0520
6,17035,PERNOD RICARD USA,"2,773,367.7300","596,082.6500",0.2149,0.9837,1.2738,0.0061,0.0046,0.0649,0.0566
7,3960,DIAGEO NORTH AMERICA INC,"2,640,491.1900","736,751.6400",0.2790,0.9853,1.3870,0.0058,0.0057,0.0708,0.0622
8,3960,DIAGEO NORTH AMERICA INC,"2,592,041.3500","503,706.5100",0.1943,0.9187,1.2412,0.0057,0.0039,0.0765,0.0661
9,12546,JIM BEAM BRANDS COMPANY,"2,435,393.3900","661,966.2500",0.2718,0.9834,1.3733,0.0054,0.0051,0.0819,0.0712


In [31]:
# ============================================================
# 20. SALES VS PROFIT CONTRIBUTION
# ============================================================

total_sales = (
    vendor_performance[
        "TotalSalesDollars"
    ].sum()
)

total_profit = (
    vendor_performance[
        "GrossProfit"
    ].sum()
)

sales_profit_contribution = vendor_performance[
    [
        "VendorNumber",
        "VendorName",
        "TotalSalesDollars",
        "GrossProfit"
    ]
].copy()

sales_profit_contribution["SalesContribution"] = (
    sales_profit_contribution["TotalSalesDollars"]
    / total_sales
)

sales_profit_contribution["ProfitContribution"] = (
    sales_profit_contribution["GrossProfit"]
    / total_profit
)

sales_profit_contribution = (
    sales_profit_contribution
    .sort_values(
        "SalesContribution",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "sales_profit_contribution shape:",
    sales_profit_contribution.shape
)

display(
    sales_profit_contribution.head(10)
)

sales_profit_contribution shape: (10547, 6)


,VendorNumber,VendorName,TotalSalesDollars,GrossProfit,SalesContribution,ProfitContribution
0,1128,BROWN-FORMAN CORP,"5,101,919.5100","1,290,667.9100",0.0113,0.0099
1,4425,MARTIGNETTI COMPANIES,"4,819,073.4900","1,015,032.2700",0.0107,0.0078
2,17035,PERNOD RICARD USA,"4,538,120.6000","1,119,816.9200",0.0100,0.0086
3,3960,DIAGEO NORTH AMERICA INC,"4,475,972.8800","1,214,774.9400",0.0099,0.0093
4,3960,DIAGEO NORTH AMERICA INC,"4,223,107.6200","1,199,901.6100",0.0094,0.0092
5,480,BACARDI USA INC,"3,383,912.4000","917,276.4700",0.0075,0.0071
6,17035,PERNOD RICARD USA,"2,773,367.7300","596,082.6500",0.0061,0.0046
7,3960,DIAGEO NORTH AMERICA INC,"2,640,491.1900","736,751.6400",0.0058,0.0057
8,3960,DIAGEO NORTH AMERICA INC,"2,592,041.3500","503,706.5100",0.0057,0.0039
9,12546,JIM BEAM BRANDS COMPANY,"2,435,393.3900","661,966.2500",0.0054,0.0051


In [32]:
# ============================================================
# 21. PROFIT MARGIN DISTRIBUTION
# ============================================================

margin = (
    vendor_sales_summary[
        "ProfitMargin"
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

profit_margin_summary = margin.describe()

display(profit_margin_summary)

count   10,514.0000
mean        -0.1589
std          4.4729
min       -237.3064
25%          0.1535
50%          0.3078
75%          0.4021
max          0.9972
Name: ProfitMargin, dtype: float64

In [33]:
# ============================================================
# 22. NEGATIVE PROFITABILITY ANALYSIS
# ============================================================

df = vendor_sales_summary.copy()

df["ProfitabilityGroup"] = np.where(
    df["ProfitMargin"] < 0,
    "Negative",
    "Non-Negative"
)

negative_profitability = (
    df.groupby(
        "ProfitabilityGroup"
    )
    .agg(
        Records=(
            "ProfitMargin",
            "size"
        ),
        MedianStockTurnover=(
            "StockTurnover",
            "median"
        ),
        MedianSalesToPurchaseRatio=(
            "SalesToPurchaseRatio",
            "median"
        )
    )
    .reset_index()
)

display(negative_profitability)

,ProfitabilityGroup,Records,MedianStockTurnover,MedianSalesToPurchaseRatio
0,Negative,1949,0.4444,0.6416
1,Non-Negative,8743,1.0000,1.5006


In [34]:
# ============================================================
# 23. PREPARE VENDOR EFFICIENCY DATASET
# ============================================================

vendor_efficiency = vendor_performance[
    [
        "VendorNumber",
        "VendorName",
        "TotalSalesDollars",
        "GrossProfit",
        "ProfitMargin",
        "StockTurnover",
        "SalesToPurchaseRatio"
    ]
].copy()

print(
    "vendor_efficiency shape:",
    vendor_efficiency.shape
)

display(vendor_efficiency.head(10))

vendor_efficiency shape: (10547, 7)


,VendorNumber,VendorName,TotalSalesDollars,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio
0,1128,BROWN-FORMAN CORP,"5,101,919.5100","1,290,667.9100",0.2530,0.9791,1.3386
1,4425,MARTIGNETTI COMPANIES,"4,819,073.4900","1,015,032.2700",0.2106,0.9769,1.2668
2,17035,PERNOD RICARD USA,"4,538,120.6000","1,119,816.9200",0.2468,0.9986,1.3276
3,3960,DIAGEO NORTH AMERICA INC,"4,475,972.8800","1,214,774.9400",0.2714,0.9937,1.3725
4,3960,DIAGEO NORTH AMERICA INC,"4,223,107.6200","1,199,901.6100",0.2841,0.9836,1.3969
5,480,BACARDI USA INC,"3,383,912.4000","917,276.4700",0.2711,1.0220,1.3719
6,17035,PERNOD RICARD USA,"2,773,367.7300","596,082.6500",0.2149,0.9837,1.2738
7,3960,DIAGEO NORTH AMERICA INC,"2,592,041.3500","503,706.5100",0.1943,0.9187,1.2412
8,3960,DIAGEO NORTH AMERICA INC,"2,640,491.1900","736,751.6400",0.2790,0.9853,1.3870
9,12546,JIM BEAM BRANDS COMPANY,"2,435,393.3900","661,966.2500",0.2718,0.9834,1.3733


In [35]:
# ============================================================
# 24. CLEAN VENDOR EFFICIENCY DATASET
# ============================================================

vendor_efficiency_clean = (
    vendor_efficiency
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
    .reset_index(drop=True)
)

print(
    "vendor_efficiency_clean shape:",
    vendor_efficiency_clean.shape
)

display(
    vendor_efficiency_clean.head(10)
)

vendor_efficiency_clean shape: (10507, 7)


,VendorNumber,VendorName,TotalSalesDollars,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio
0,1128,BROWN-FORMAN CORP,"5,101,919.5100","1,290,667.9100",0.2530,0.9791,1.3386
1,4425,MARTIGNETTI COMPANIES,"4,819,073.4900","1,015,032.2700",0.2106,0.9769,1.2668
2,17035,PERNOD RICARD USA,"4,538,120.6000","1,119,816.9200",0.2468,0.9986,1.3276
3,3960,DIAGEO NORTH AMERICA INC,"4,475,972.8800","1,214,774.9400",0.2714,0.9937,1.3725
4,3960,DIAGEO NORTH AMERICA INC,"4,223,107.6200","1,199,901.6100",0.2841,0.9836,1.3969
5,480,BACARDI USA INC,"3,383,912.4000","917,276.4700",0.2711,1.0220,1.3719
6,17035,PERNOD RICARD USA,"2,773,367.7300","596,082.6500",0.2149,0.9837,1.2738
7,3960,DIAGEO NORTH AMERICA INC,"2,592,041.3500","503,706.5100",0.1943,0.9187,1.2412
8,3960,DIAGEO NORTH AMERICA INC,"2,640,491.1900","736,751.6400",0.2790,0.9853,1.3870
9,12546,JIM BEAM BRANDS COMPANY,"2,435,393.3900","661,966.2500",0.2718,0.9834,1.3733


In [36]:
# ============================================================
# 25. EFFICIENCY CORRELATION ANALYSIS
# ============================================================

efficiency_correlations = (
    vendor_efficiency_clean[
        [
            "ProfitMargin",
            "StockTurnover",
            "SalesToPurchaseRatio"
        ]
    ]
    .corr()
)

display(efficiency_correlations)

,ProfitMargin,StockTurnover,SalesToPurchaseRatio
ProfitMargin,1.0000,0.0489,0.0512
StockTurnover,0.0489,1.0000,0.9971
SalesToPurchaseRatio,0.0512,0.9971,1.0000


In [37]:
# ============================================================
# 26. HIGH-SALES OPERATIONAL EFFICIENCY SCREENING
# ============================================================

sales_threshold = 29524.25
stock_turnover_threshold = 0.983431
sales_purchase_ratio_threshold = 1.444635

high_sales_efficient = vendor_efficiency_clean[
    (
        vendor_efficiency_clean[
            "TotalSalesDollars"
        ] >= sales_threshold
    )
    &
    (
        vendor_efficiency_clean[
            "StockTurnover"
        ] >= stock_turnover_threshold
    )
    &
    (
        vendor_efficiency_clean[
            "SalesToPurchaseRatio"
        ] >= sales_purchase_ratio_threshold
    )
].copy()

high_sales_efficient = (
    high_sales_efficient
    .sort_values(
        "TotalSalesDollars",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "high_sales_efficient shape:",
    high_sales_efficient.shape
)

display(high_sales_efficient.head(10))

high_sales_efficient shape: (882, 7)


,VendorNumber,VendorName,TotalSalesDollars,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio
0,10000,MAJESTIC FINE WINES,"2,326,007.7800","865,935.4200",0.3723,0.9934,1.5931
1,480,BACARDI USA INC,"2,189,368.7800","711,367.3600",0.3249,1.0162,1.4813
2,3960,DIAGEO NORTH AMERICA INC,"2,068,091.1600","671,960.3900",0.3249,1.0146,1.4813
3,480,BACARDI USA INC,"1,657,809.5900","550,932.9800",0.3323,1.0085,1.4977
4,8112,MOET HENNESSY USA INC,"1,533,634.5200","519,150.2300",0.3385,0.9889,1.5117
5,1128,BROWN-FORMAN CORP,"1,255,188.9400","411,982.1800",0.3282,1.0052,1.4886
6,3960,DIAGEO NORTH AMERICA INC,"1,210,481.3400","499,561.4400",0.4127,1.2611,1.7027
7,1392,CONSTELLATION BRANDS INC,"1,016,328.6100","404,000.2000",0.3975,0.9980,1.6598
8,6785,PALM BAY INTERNATIONAL INC,"945,345.0200","346,269.0200",0.3663,0.9912,1.5780
9,1128,BROWN-FORMAN CORP,"937,915.3600","292,044.0000",0.3114,0.9849,1.4522


In [38]:
# ============================================================
# 27. HIGH-SALES WEAK-EFFICIENCY SCREENING
# ============================================================

high_sales_weak = vendor_efficiency_clean[
    (
        vendor_efficiency_clean[
            "TotalSalesDollars"
        ] >= sales_threshold
    )
    &
    (
        (
            vendor_efficiency_clean[
                "StockTurnover"
            ]
            < stock_turnover_threshold
        )
        |
        (
            vendor_efficiency_clean[
                "SalesToPurchaseRatio"
            ]
            < sales_purchase_ratio_threshold
        )
    )
].copy()

high_sales_weak = (
    high_sales_weak
    .sort_values(
        "TotalSalesDollars",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "high_sales_weak shape:",
    high_sales_weak.shape
)

display(high_sales_weak.head(10))

high_sales_weak shape: (1747, 7)


,VendorNumber,VendorName,TotalSalesDollars,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio
0,1128,BROWN-FORMAN CORP,"5,101,919.5100","1,290,667.9100",0.2530,0.9791,1.3386
1,4425,MARTIGNETTI COMPANIES,"4,819,073.4900","1,015,032.2700",0.2106,0.9769,1.2668
2,17035,PERNOD RICARD USA,"4,538,120.6000","1,119,816.9200",0.2468,0.9986,1.3276
3,3960,DIAGEO NORTH AMERICA INC,"4,475,972.8800","1,214,774.9400",0.2714,0.9937,1.3725
4,3960,DIAGEO NORTH AMERICA INC,"4,223,107.6200","1,199,901.6100",0.2841,0.9836,1.3969
5,480,BACARDI USA INC,"3,383,912.4000","917,276.4700",0.2711,1.0220,1.3719
6,17035,PERNOD RICARD USA,"2,773,367.7300","596,082.6500",0.2149,0.9837,1.2738
7,3960,DIAGEO NORTH AMERICA INC,"2,640,491.1900","736,751.6400",0.2790,0.9853,1.3870
8,3960,DIAGEO NORTH AMERICA INC,"2,592,041.3500","503,706.5100",0.1943,0.9187,1.2412
9,12546,JIM BEAM BRANDS COMPANY,"2,435,393.3900","661,966.2500",0.2718,0.9834,1.3733


In [39]:
# ============================================================
# 28. BRAND-LEVEL ANALYSIS
# ============================================================

brand_analysis = (
    vendor_sales_summary
    .groupby(
        "Brand",
        as_index=False
    )
    .agg(
        TotalSalesDollars=(
            "TotalSalesDollars",
            "sum"
        ),
        GrossProfit=(
            "GrossProfit",
            "sum"
        )
    )
)

brand_analysis["ProfitMargin"] = np.where(
    brand_analysis[
        "TotalSalesDollars"
    ] != 0,
    brand_analysis[
        "GrossProfit"
    ]
    / brand_analysis[
        "TotalSalesDollars"
    ],
    np.nan
)

print(
    "brand_analysis shape:",
    brand_analysis.shape
)

display(
    brand_analysis.head(10)
)

brand_analysis shape: (10663, 4)


,Brand,TotalSalesDollars,GrossProfit,ProfitMargin
0,58,"43,341.5400","10,397.5400",0.2399
1,60,"18,716.2500","6,632.0500",0.3543
2,61,"4,364.8800","1,057.6800",0.2423
3,62,"119,863.7500","28,119.7500",0.2346
4,63,"112,249.2200","25,285.9200",0.2253
5,70,349.8600,154.4200,0.4414
6,72,"17,325.3100","5,393.0400",0.3113
7,75,314.7900,205.3900,0.6525
8,77,"143,416.7100","29,833.2300",0.2080
9,79,"78,923.1400","23,329.1600",0.2956


In [41]:
# ============================================================
# 23. BRAND SEGMENTATION FUNCTION
# ============================================================

def segment_brands(
    brand_analysis,
    low_sales_threshold,
    high_sales_threshold,
    high_margin_threshold,
    low_margin_threshold
):
    """
    Segment brands based on established sales-volume
    and profit-margin thresholds.
    """

    result = brand_analysis.copy()

    result["Segment"] = "Other"

    result.loc[
        (
            result["TotalSalesDollars"]
            <= low_sales_threshold
        )
        &
        (
            result["ProfitMargin"]
            >= high_margin_threshold
        ),
        "Segment"
    ] = "Low Sales / High Margin"

    result.loc[
        (
            result["TotalSalesDollars"]
            >= high_sales_threshold
        )
        &
        (
            result["ProfitMargin"]
            <= low_margin_threshold
        ),
        "Segment"
    ] = "High Sales / Low Margin"

    return result

In [42]:
# ============================================================
# 23. BRAND SEGMENTATION
# ============================================================

brand_segments = segment_brands(
    brand_analysis,
    low_sales_threshold=729.27,
    high_sales_threshold=28459.39,
    high_margin_threshold=0.402028,
    low_margin_threshold=0.1535
)

print(
    "brand_segments shape:",
    brand_segments.shape
)

display(
    brand_segments.head(10)
)

brand_segments shape: (10663, 5)


,Brand,TotalSalesDollars,GrossProfit,ProfitMargin,Segment
0,58,"43,341.5400","10,397.5400",0.2399,Other
1,60,"18,716.2500","6,632.0500",0.3543,Other
2,61,"4,364.8800","1,057.6800",0.2423,Other
3,62,"119,863.7500","28,119.7500",0.2346,Other
4,63,"112,249.2200","25,285.9200",0.2253,Other
5,70,349.8600,154.4200,0.4414,Low Sales / High Margin
6,72,"17,325.3100","5,393.0400",0.3113,Other
7,75,314.7900,205.3900,0.6525,Low Sales / High Margin
8,77,"143,416.7100","29,833.2300",0.2080,Other
9,79,"78,923.1400","23,329.1600",0.2956,Other


In [43]:
# ============================================================
# 23B. BRAND SEGMENT DISTRIBUTION
# ============================================================

brand_segment_counts = (
    brand_segments["Segment"]
    .value_counts()
    .rename_axis("Segment")
    .reset_index(name="Count")
)

display(brand_segment_counts)

,Segment,Count
0,Other,9975
1,Low Sales / High Margin,482
2,High Sales / Low Margin,206


In [44]:
# ============================================================
# 24. DESCRIPTION-LEVEL ANALYSIS
# ============================================================

def prepare_description_analysis(
    vendor_sales_summary
):
    """
    Aggregate vendor-brand observations to
    product-description level.

    ProfitMargin is calculated after aggregation.
    """

    description_analysis = (
        vendor_sales_summary
        .groupby(
            "Description",
            as_index=False
        )
        .agg(
            TotalSalesDollars=(
                "TotalSalesDollars",
                "sum"
            ),
            GrossProfit=(
                "GrossProfit",
                "sum"
            )
        )
    )

    description_analysis["ProfitMargin"] = np.where(
        description_analysis[
            "TotalSalesDollars"
        ] != 0,
        description_analysis[
            "GrossProfit"
        ]
        / description_analysis[
            "TotalSalesDollars"
        ],
        np.nan
    )

    return description_analysis

In [45]:
# ============================================================
# 24A. EXECUTE DESCRIPTION-LEVEL ANALYSIS
# ============================================================

description_analysis = prepare_description_analysis(
    vendor_sales_summary
)

print(
    "description_analysis shape:",
    description_analysis.shape
)

display(
    description_analysis.head(10)
)

description_analysis shape: (9651, 4)


,Description,TotalSalesDollars,GrossProfit,ProfitMargin
0,(RI) 1,"21,519.0900","3,886.4900",0.1806
1,.nparalleled Svgn Blanc,"1,094.6300",328.1500,0.2998
2,10 Span Cab Svgn CC,"2,703.8900",566.1300,0.2094
3,10 Span Chard CC,"3,325.5600",924.7200,0.2781
4,10 Span Pnt Gris Monterey Cy,"2,082.2200",671.0200,0.3223
5,10 Span Pnt Nr CC,"2,441.7400",630.8500,0.2584
6,1000 Stories Znfdl,"1,918.8000","-1,051.4400",-0.5480
7,12 Days of Pearls Gift Set,309.6900,302.5000,0.9768
8,13 Celsius Svgn Bl,"34,041.2300","19,631.5700",0.5767
9,13th Colony Sthrn Corn Whsky,359.8200,152.7300,0.4245


In [46]:
# ============================================================
# 25. DESCRIPTION-LEVEL SEGMENTATION FUNCTION
# ============================================================

def segment_descriptions(
    description_analysis,
    low_sales_threshold,
    high_sales_threshold,
    high_margin_threshold,
    low_margin_threshold
):
    """
    Segment product descriptions based on
    established sales-volume and profit-margin thresholds.
    """

    result = description_analysis.copy()

    result["Segment"] = "Other"

    result.loc[
        (
            result["TotalSalesDollars"]
            <= low_sales_threshold
        )
        &
        (
            result["ProfitMargin"]
            >= high_margin_threshold
        ),
        "Segment"
    ] = "Low Sales / High Margin"

    result.loc[
        (
            result["TotalSalesDollars"]
            >= high_sales_threshold
        )
        &
        (
            result["ProfitMargin"]
            <= low_margin_threshold
        ),
        "Segment"
    ] = "High Sales / Low Margin"

    return result

In [47]:
# ============================================================
# 25A. EXECUTE DESCRIPTION-LEVEL SEGMENTATION
# ============================================================

description_segments = segment_descriptions(
    description_analysis,
    low_sales_threshold=729.27,
    high_sales_threshold=28459.39,
    high_margin_threshold=0.402028,
    low_margin_threshold=0.1535
)

print(
    "description_segments shape:",
    description_segments.shape
)

display(
    description_segments.head(10)
)

description_segments shape: (9651, 5)


,Description,TotalSalesDollars,GrossProfit,ProfitMargin,Segment
0,(RI) 1,"21,519.0900","3,886.4900",0.1806,Other
1,.nparalleled Svgn Blanc,"1,094.6300",328.1500,0.2998,Other
2,10 Span Cab Svgn CC,"2,703.8900",566.1300,0.2094,Other
3,10 Span Chard CC,"3,325.5600",924.7200,0.2781,Other
4,10 Span Pnt Gris Monterey Cy,"2,082.2200",671.0200,0.3223,Other
5,10 Span Pnt Nr CC,"2,441.7400",630.8500,0.2584,Other
6,1000 Stories Znfdl,"1,918.8000","-1,051.4400",-0.5480,Other
7,12 Days of Pearls Gift Set,309.6900,302.5000,0.9768,Low Sales / High Margin
8,13 Celsius Svgn Bl,"34,041.2300","19,631.5700",0.5767,Other
9,13th Colony Sthrn Corn Whsky,359.8200,152.7300,0.4245,Low Sales / High Margin


In [48]:
# ============================================================
# 25B. DESCRIPTION SEGMENT DISTRIBUTION
# ============================================================

description_segment_counts = (
    description_segments["Segment"]
    .value_counts()
    .rename_axis("Segment")
    .reset_index(name="Count")
)

display(description_segment_counts)

,Segment,Count
0,Other,9026
1,Low Sales / High Margin,436
2,High Sales / Low Margin,189


In [49]:
# ============================================================
# 25C. DESCRIPTION-LEVEL THRESHOLD DIAGNOSTIC
# ============================================================

print("Description Analysis:")
print("Rows:", len(description_analysis))

print("\nSales quantiles:")
display(
    description_analysis["TotalSalesDollars"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

print("\nProfit Margin quantiles:")
display(
    description_analysis["ProfitMargin"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

Description Analysis:
Rows: 9651

Sales quantiles:


count       9,651.0000
mean       46,795.1698
std       244,521.3992
min             0.0000
10%           164.9500
25%           735.3100
50%         5,028.6400
75%        25,762.6900
90%        84,969.4100
max     7,964,746.7600
Name: TotalSalesDollars, dtype: float64


Profit Margin quantiles:


count   9,483.0000
mean       -0.1853
std         4.6906
min      -237.3064
10%        -0.5107
25%         0.1452
50%         0.3109
75%         0.4057
90%         0.7115
max         0.9960
Name: ProfitMargin, dtype: float64